In [1]:
import sys
import git
import pathlib

# Set up the PROJ_ROOT variable
PROJ_ROOT_PATH = pathlib.Path(git.Repo('.', search_parent_directories=True).working_tree_dir)
PROJ_ROOT =  str(PROJ_ROOT_PATH)
if PROJ_ROOT not in sys.path:
    sys.path.append(PROJ_ROOT)

# Explicitly add the current notebook's directory
CURRENT_DIR = str(pathlib.Path().absolute())
if CURRENT_DIR not in sys.path:
    sys.path.insert(0, CURRENT_DIR)

In [2]:
import numpy as np
from scipy.integrate import quad
import math
import matplotlib.pyplot as plt
from library.utils import fontstyle
title_font, axis_label_font, tick_label_font, legend_font, text_font = fontstyle

# Fridge Details

In [3]:
from library.fridges import TEMP_STAGES, FRIDGE_LIBRARY
# Assuming XLD1000sl
# We will use the operating temperatures and cable lengths corresponding to an XLD1000sl fridge
fridge = FRIDGE_LIBRARY["XLD1000SL"]
cooling_power_budget = fridge["cooling_power"]
operating_temp = fridge["temp"]
flange_separation = fridge["lengths"]

In [4]:
from library.cables import create_cable_instance, CABLE_REGISTRY

In [5]:
_SI_PREFIX = {
    -24: "y", -21: "z", -18: "a", -15: "f", -12: "p", -9: "n",
    -6: "u", -3: "m", 0: "", 3: "k", 6: "M", 9: "G", 12: "T",
    15: "P", 18: "E", 21: "Z", 24: "Y",
}

def fmt_eng(x: float, sig: int = 3, unit: str = "W") -> str:
    """Engineering notation with SI prefix; exponent divisible by 3."""
    if x is None: 
        return None
    else:
        x = float(x)
        if x == 0.0 or not np.isfinite(x):
            return f"{x:g}{unit}"
    
        exp3 = int(np.floor(np.log10(abs(x)) / 3) * 3)
        exp3 = max(min(exp3, 24), -24)  # clamp to known prefixes
        scaled = x / (10 ** exp3)
        prefix = _SI_PREFIX[exp3]
    
        # sig significant figures, no trailing clutter
        s = f"{scaled:.2f}".rstrip('0').rstrip('.')
        # return rf"\qty{{{s}}}{prefix}{unit}".strip()
        return f"{s} {prefix}{unit}".strip()

In [6]:
cable_list =list(CABLE_REGISTRY.keys())

In [7]:
PHL_DATA = {}
PHL_DATA_RAW = {}

for cable_name in cable_list:
    PHL_DATA[cable_name] = {}
    PHL_DATA_RAW[cable_name] = {}
    for temp_stage in TEMP_STAGES[1:]:
        # create cable instance
        cable = create_cable_instance(cable_name)
        
        # get cable length
        cable_length = flange_separation[temp_stage]
    
        # get thermal gradient
        current_temp_stage_idx = TEMP_STAGES.index(temp_stage)
        if current_temp_stage_idx > 0:
            prev_temp_stage = TEMP_STAGES[current_temp_stage_idx - 1]
    
        T_lo = operating_temp[temp_stage]
        T_hi = operating_temp[prev_temp_stage]
        
        phl = cable.get_PHL(temp_stage, cable_length, T_lo, T_hi)

        PHL_DATA[cable_name][temp_stage] = fmt_eng(phl)
        PHL_DATA_RAW[cable_name][temp_stage] = phl

In [8]:
import pandas as pd

df = pd.DataFrame.from_dict(PHL_DATA, orient='index')
print(df)

                           50K         4K      Still          CP         MXC
SS_Drive              30.06 mW   822.3 uW    2.52 uW   391.76 nW    13.89 nW
SS_Flux                37.4 mW  986.76 uW    1.26 uW   293.82 nW    30.99 nW
NbTi_coax                  NaN        NaN  630.63 nW   293.82 nW    21.37 nW
SC_086_NbTi_coax           NaN        NaN  155.12 nW    25.26 nW   347.78 pW
GHOST                      NaN        NaN        NaN         NaN         NaN
Cu_35_bias            10.02 mW    1.23 mW        NaN         NaN         NaN
Ag                     2.94 mW  643.84 uW    1.68 uW   495.88 nW    12.01 nW
NbTi                       NaN        NaN  168.27 nW    49.59 nW      1.2 nW
HDW                    7.57 mW   315.7 uW    2.37 uW    549.1 nW    11.32 nW
HEMT_Bias_Cu          59.63 mW   20.42 mW  254.49 uW   117.74 uW    12.22 uW
Fiber                 34.03 uW    1.13 uW   17.98 nW      4.4 nW    21.87 pW
HEMT_Bias_Mn           2.08 mW   76.41 uW  328.14 nW  -104.26 nW   -39.06 nW

In [9]:
df_raw = pd.DataFrame.from_dict(PHL_DATA_RAW, orient='index')
with pd.option_context('display.float_format', lambda x: f'{x:.6e}'):
    print(df_raw)

                             50K           4K        Still            CP  \
SS_Drive            3.005722e-02 8.223020e-04 2.522523e-06  3.917615e-07   
SS_Flux             3.740455e-02 9.867624e-04 1.261261e-06  2.938211e-07   
NbTi_coax                    NaN          NaN 6.306306e-07  2.938211e-07   
SC_086_NbTi_coax             NaN          NaN 1.551208e-07  2.525724e-08   
GHOST                        NaN          NaN          NaN           NaN   
Cu_35_bias          1.001907e-02 1.233453e-03          NaN           NaN   
Ag                  2.943203e-03 6.438366e-04 1.682671e-06  4.958779e-07   
NbTi                         NaN          NaN 1.682671e-07  4.958779e-08   
HDW                 7.566000e-03 3.157000e-04 2.373000e-06  5.491000e-07   
HEMT_Bias_Cu        5.962563e-02 2.042272e-02 2.544851e-04  1.177374e-04   
Fiber               3.402993e-05 1.130479e-06 1.797569e-08  4.399751e-09   
HEMT_Bias_Mn        2.081771e-03 7.641043e-05 3.281391e-07 -1.042638e-07   
HEMT_Bias_YB